# fishi — Grounded SAM experiments

End-to-end on Colab (use a **GPU** runtime):
1. installs fishi,
2. downloads WoodScape into Drive if needed,
3. runs the preprocessing x pipeline grid for Grounding DINO + SAM 1/2 on the test split,
4. writes one metrics JSON per cell into a shared `metrics/` folder on Drive, and a CSV.

OpenWorldSAM runs in its own notebook/runtime and writes into the same `metrics/`
folder, so no merge is needed. Keep `DRIVE_ROOT` the same across both.

In [ ]:
!pip install -q "fishi[colab] @ git+https://github.com/AndreKoraleski/fishi.git"

In [ ]:
import torch
from google.colab import drive

from fishi.evaluation import run
from fishi.preprocess import Identity, Patches, Rectify
from fishi.report import save_cell, to_csv
from fishi.segmentation.grounded_sam import GroundedSam1, GroundedSam2
from fishi.woodscape import classes
from fishi.woodscape.config import get_settings
from fishi.woodscape.dataset import WoodScapeDataset
from fishi.woodscape.download import download_woodscape
from fishi.woodscape.splits import split_datasets

DRIVE_ROOT = "/content/drive/MyDrive/fishi"
DATA_DIRECTORY = f"{DRIVE_ROOT}/woodscape"
CACHE_DIRECTORY = f"{DRIVE_ROOT}/cache"
METRICS_DIRECTORY = f"{DRIVE_ROOT}/metrics"

drive.mount("/content/drive")
settings = get_settings(data_directory=DATA_DIRECTORY)

In [ ]:
download_woodscape(settings)  # resumable: skips files already on Drive

In [ ]:
test = split_datasets(WoodScapeDataset(settings))["test"]
print("test samples:", len(test), "| cuda:", torch.cuda.is_available())

In [ ]:
for build_pipeline in (GroundedSam1, GroundedSam2):  # largest checkpoints by default
    pipeline = build_pipeline()
    for processor in (Identity(), Rectify(), Patches()):
        metrics = run(
            processor, pipeline, test, classes.PROMPTS,
            len(classes.CLASS_NAMES), cache_directory=CACHE_DIRECTORY,
        )
        save_cell(metrics, classes.CLASS_NAMES, pipeline.name, processor.name, METRICS_DIRECTORY)
        print(pipeline.name, processor.name, "mIoU", round(float(metrics["miou"]), 4))
    del pipeline
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd

to_csv(METRICS_DIRECTORY, f"{DRIVE_ROOT}/metrics.csv")
pd.read_csv(f"{DRIVE_ROOT}/metrics.csv")